# 00. 환경 설정 & 인프라 확인

노트북을 실행하기 전, 환경이 제대로 구성되어 있는지 확인합니다.

- Python 버전 & 가상환경
- 핵심 패키지 버전 (pandas, pyarrow, google-cloud-bigquery)
- uv 프로젝트 설정
- GCP 서비스 계정 키 & BigQuery 연결
- 디렉토리 구조

## 1. Python & 가상환경

In [1]:
import sys
import os

print(f"Python version: {sys.version}")
print(f"Executable:     {sys.executable}")
print(f"Prefix:         {sys.prefix}")
print(f"In virtualenv:  {sys.prefix != sys.base_prefix}")

Python version: 3.13.11 (main, Dec  9 2025, 20:26:22) [Clang 21.1.4 ]
Executable:     /Users/kakao/bda-2026-01/.venv/bin/python
Prefix:         /Users/kakao/bda-2026-01/.venv
In virtualenv:  True


## 2. 핵심 패키지 버전

In [2]:
import importlib.metadata

PACKAGES = [
    "pandas",
    "pyarrow",
    "google-cloud-bigquery",
    "google-cloud-bigquery-storage",
    "db-dtypes",
    "matplotlib",
    "requests",
    "tqdm",
    "ipykernel",
    "nbconvert",
]

print(f"{'Package':<35s} {'Version':<15s}")
print("-" * 50)
for pkg in PACKAGES:
    try:
        ver = importlib.metadata.version(pkg)
        print(f"{pkg:<35s} {ver:<15s}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg:<35s} {'NOT INSTALLED':<15s}")

Package                             Version        
--------------------------------------------------
pandas                              3.0.1          
pyarrow                             23.0.1         
google-cloud-bigquery               3.40.1         
google-cloud-bigquery-storage       2.36.2         
db-dtypes                           1.4.4          
matplotlib                          3.10.8         
requests                            2.32.5         
tqdm                                4.67.3         
ipykernel                           7.2.0          
nbconvert                           7.17.0         


## 3. uv 프로젝트 설정

In [3]:
from pathlib import Path

PROJECT_ROOT = Path("../../")

# pyproject.toml 확인
pyproject = PROJECT_ROOT / "pyproject.toml"
assert pyproject.exists(), "pyproject.toml not found!"
print("pyproject.toml:")
print(pyproject.read_text())

# .python-version 확인
python_version_file = PROJECT_ROOT / ".python-version"
if python_version_file.exists():
    print(f"\n.python-version: {python_version_file.read_text().strip()}")

# uv.lock 존재 여부
uv_lock = PROJECT_ROOT / "uv.lock"
print(f"\nuv.lock exists: {uv_lock.exists()}")
if uv_lock.exists():
    print(f"uv.lock size: {uv_lock.stat().st_size / 1024:.1f} KB")

pyproject.toml:
[project]
name = "bda-2"
version = "0.1.0"
description = "Add your description here"
readme = "README.md"
requires-python = ">=3.13"
dependencies = [
    "db-dtypes>=1.4.4",
    "google-cloud-bigquery>=3.40.1",
    "google-cloud-bigquery-storage>=2.36.2",
    "ipykernel>=7.2.0",
    "matplotlib>=3.10.8",
    "nbconvert>=7.17.0",
    "pandas>=3.0.1",
    "pyarrow>=23.0.1",
    "requests>=2.32.5",
    "tqdm>=4.67.3",
]

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[tool.hatch.build.targets.wheel]
packages = ["src/gharchive", "src/ghrec"]


.python-version: 3.13

uv.lock exists: True
uv.lock size: 256.3 KB


## 4. GCP 서비스 계정 & BigQuery 연결

In [4]:
import json

KEY_PATH = os.environ.get("GCP_KEY_PATH", "gcp-key.json")
key_file = Path(KEY_PATH)

print(f"Key path: {KEY_PATH}")
print(f"Key exists: {key_file.exists()}")

if key_file.exists():
    with open(key_file) as f:
        key_data = json.load(f)
    print(f"Project ID: {key_data.get('project_id', 'N/A')}")
    print(f"Client email: {key_data.get('client_email', 'N/A')}")
    print(f"Key type: {key_data.get('type', 'N/A')}")
else:
    print("\n⚠ gcp-key.json이 없습니다. BigQuery 노트북(01, ghrec/01, ghrec/03)을 실행할 수 없습니다.")
    print("GCP Console에서 서비스 계정 키를 다운로드하세요.")

Key path: gcp-key.json
Key exists: True
Project ID: bda-coai
Client email: dane-gcp@bda-coai.iam.gserviceaccount.com
Key type: service_account


In [5]:
# BigQuery 연결 테스트
from gharchive.client import create_client

if key_file.exists():
    client = create_client(KEY_PATH)
    print(f"BigQuery client ready!")
    print(f"Project: {client.project}")

    # 간단한 쿼리 테스트
    test_query = "SELECT 1 AS test"
    result = client.query(test_query).to_dataframe()
    print(f"Test query OK: {result['test'].iloc[0]}")
else:
    print("BigQuery 연결 테스트 SKIP (키 파일 없음)")

BigQuery client ready!
Project: bda-coai


Test query OK: 1


## 5. 디렉토리 구조

In [6]:
# 프로젝트 디렉토리 구조 확인
def show_tree(path: Path, prefix: str = "", max_depth: int = 3, _depth: int = 0):
    """Print directory tree."""
    if _depth >= max_depth:
        return
    entries = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name))
    entries = [e for e in entries if not e.name.startswith(".") and e.name != "__pycache__"]
    for i, entry in enumerate(entries):
        connector = "└── " if i == len(entries) - 1 else "├── "
        print(f"{prefix}{connector}{entry.name}")
        if entry.is_dir():
            extension = "    " if i == len(entries) - 1 else "│   "
            show_tree(entry, prefix + extension, max_depth, _depth + 1)

print("Project structure:")
show_tree(PROJECT_ROOT.resolve())

Project structure:
├── data
│   ├── _format_bench
│   │   ├── Int32_object.csv
│   │   ├── Int32_object.jsonl
│   │   ├── Int32_object.parquet
│   │   ├── all_28days.csv
│   │   ├── all_28days.jsonl
│   │   ├── dtype_Int32.parquet
│   │   ├── dtype_Int32_cat.parquet
│   │   ├── dtype_Int32_cnt16.parquet
│   │   ├── dtype_int64.parquet
│   │   ├── optimized_Int32_cat.csv
│   │   ├── optimized_Int32_cat.jsonl
│   │   ├── optimized_Int32_cat.parquet
│   │   ├── raw_int64.csv
│   │   ├── raw_int64.jsonl
│   │   └── raw_int64.parquet
│   ├── daily_agg
│   │   ├── 20260215.parquet
│   │   ├── 20260216.parquet
│   │   ├── 20260217.parquet
│   │   ├── 20260218.parquet
│   │   ├── 20260219.parquet
│   │   ├── 20260220.parquet
│   │   ├── 20260221.parquet
│   │   ├── 20260222.parquet
│   │   ├── 20260223.parquet
│   │   ├── 20260224.parquet
│   │   ├── 20260225.parquet
│   │   ├── 20260226.parquet
│   │   ├── 20260227.parquet
│   │   ├── 20260228.parquet
│   │   ├── 20260301.parquet
│   │   ├── 

## 6. GitHub Token (선택)

`ghrec/03_repo_metadata` 노트북에서 GitHub REST API를 사용합니다.  
토큰 없이도 동작하지만, rate limit이 60회/시간으로 제한됩니다.

`gh auth token`으로 토큰이 설정되어 있는지 확인합니다.

In [7]:
import subprocess

result = subprocess.run(["gh", "auth", "token"], capture_output=True, text=True)
gh_token = result.stdout.strip()

if gh_token:
    print(f"gh CLI token: set ({gh_token[:8]}...)")
else:
    print("gh CLI token: not set")
    print("  → gh auth login 으로 설정하거나")
    print("  → GITHUB_TOKEN 환경변수를 설정하세요")

gh CLI token: set (gho_6c5m...)


## All Good!

위 셀들이 에러 없이 실행되면 환경 준비 완료입니다.  
다음 노트북으로 진행하세요: `01_extract_daily_agg.ipynb`